In [13]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP

In [14]:
numero_controle = "0000386-2"

data_emissao = "29/11/2024"
data_posicao = "30/04/2026"

valor_aplicado = Decimal("700000")
percentual_cdi = Decimal("115") / Decimal("100")

valor_esperado_extrato = Decimal("868347.90")

In [15]:
def parse_data_br(data):
    return datetime.strptime(data, "%d/%m/%Y").date()


def parse_data_iso(data):
    return datetime.strptime(data, "%Y-%m-%d").date()


def dinheiro(valor):
    return Decimal(valor).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def eh_dia_de_rendimento(data):
    return data in cdi_por_data

In [16]:
caminho_cdi = Path("data/taxas/cdi.json")

with caminho_cdi.open("r", encoding="utf-8") as arquivo:
    dados_cdi = json.load(arquivo)

cdi_por_data = {}

for item in dados_cdi:
    data = parse_data_iso(item["data"])
    valor = Decimal(str(item["valor"]))
    cdi_por_data[data] = valor

len(cdi_por_data), min(cdi_por_data), max(cdi_por_data)

(490, datetime.date(2024, 5, 27), datetime.date(2026, 5, 7))

In [17]:
for data_txt in ["27/04/2026", "28/04/2026", "29/04/2026", "30/04/2026"]:
    data = parse_data_br(data_txt)
    print(data_txt, cdi_por_data.get(data))

27/04/2026 0.054266
28/04/2026 0.054266
29/04/2026 0.054266
30/04/2026 0.053400


In [ ]:
inicio = parse_data_br(data_emissao)
fim = parse_data_br(data_posicao)

saldo = valor_aplicado
linhas = []

data = inicio

while data <= fim:
    saldo_abertura = saldo

    if data != inicio and eh_dia_de_rendimento(data):
        cdi_dia_percentual = cdi_por_data.get(data)

        if cdi_dia_percentual is None:
            juros = Decimal("0")
            observacao = "CDI não encontrado"
        else:
            taxa_base_decimal = cdi_dia_percentual / Decimal("100")
            taxa_aplicada_decimal = taxa_base_decimal * percentual_cdi
            juros = saldo * taxa_aplicada_decimal
            saldo = saldo + juros
            observacao = ""
    else:
        cdi_dia_percentual = None
        taxa_base_decimal = Decimal("0")
        taxa_aplicada_decimal = Decimal("0")
        juros = Decimal("0")
        observacao = "sem rendimento"

    linhas.append({
        "data": data,
        "dia_util": eh_dia_de_rendimento(data),
        "saldo_abertura": float(saldo_abertura),
        "cdi_percentual_dia": float(cdi_dia_percentual) if cdi_dia_percentual is not None else None,
        "percentual_cdi": float(percentual_cdi),
        "taxa_aplicada_percentual_dia": float(taxa_aplicada_decimal * Decimal("100")),
        "juros": float(juros),
        "saldo_fechamento": float(saldo),
        "observacao": observacao,
    })

    data += timedelta(days=1)

df = pd.DataFrame(linhas)
df.tail(10)


Valor calculado: 868462.39
Valor esperado: 868347.90


In [19]:
saldo_calculado = dinheiro(Decimal(str(df.iloc[-1]["saldo_fechamento"])))
diferenca = saldo_calculado - valor_esperado_extrato

print("Saldo calculado:", saldo_calculado)
print("Saldo esperado extrato:", valor_esperado_extrato)
print("Diferença:", diferenca)

Saldo calculado: 868462.39
Saldo esperado extrato: 868347.90
Diferença: 114.49


In [20]:
df[df["data"] >= parse_data_br("24/04/2026")]

,data,dia_util,saldo_abertura,cdi_percentual_dia,percentual_cdi,taxa_aplicada_percentual_dia,juros,saldo_fechamento,observacao
511,2026-04-24,True,865766.216891,0.054266,1.15,0.062406,540.289200,866306.506091,
512,2026-04-25,False,866306.506091,NaN,1.15,0.000000,0.000000,866306.506091,sem rendimento
513,2026-04-26,False,866306.506091,NaN,1.15,0.000000,0.000000,866306.506091,sem rendimento
514,2026-04-27,True,866306.506091,0.054266,1.15,0.062406,540.626372,866847.132463,
515,2026-04-28,True,866847.132463,0.054266,1.15,0.062406,540.963755,867388.096217,
516,2026-04-29,True,867388.096217,0.054266,1.15,0.062406,541.301348,867929.397565,
517,2026-04-30,True,867929.397565,0.053400,1.15,0.061410,532.995443,868462.393008,


In [ ]:
saldo_29 = Decimal(str(df.loc[df["data"] == parse_data_br("29/04/2026"), "saldo_fechamento"].iloc[0]))

juros_esperado_30 = valor_esperado_extrato - dinheiro(saldo_29)

taxa_aplicada_implicita = juros_esperado_30 / saldo_29
cdi_implicito = taxa_aplicada_implicita / percentual_cdi

print("Saldo em 29/04:", dinheiro(saldo_29))
print("Juros esperado em 30/04:", dinheiro(juros_esperado_30))
print("Taxa aplicada implícita:", taxa_aplicada_implicita * Decimal("100"), "% ao dia")
print("CDI implícito:", cdi_implicito * Decimal("100"), "% ao dia")

Saldo em 29/04: 867929.40
Juros esperado em 30/04: 418.50
Taxa aplicada implícita: 0.04821820774524300337348461270 % ao dia
CDI implícito: 0.04192887630021130728129096757 % ao dia


In [22]:
data_30 = parse_data_br("30/04/2026")
print("CDI no JSON em 30/04:", cdi_por_data.get(data_30), "% ao dia")

CDI no JSON em 30/04: 0.053400 % ao dia


In [ ]:
def calcular_com_deslocamento_cdi(deslocamento_dias=0, arredondar_diariamente=False):
    saldo = valor_aplicado
    linhas = []

    data = inicio

    while data <= fim:
        saldo_abertura = saldo

        if data != inicio and eh_dia_de_rendimento(data):
            data_cdi = data - timedelta(days=deslocamento_dias)

            while data_cdi not in cdi_por_data and data_cdi >= inicio:
                data_cdi -= timedelta(days=1)

            cdi_dia_percentual = cdi_por_data.get(data_cdi)

            if cdi_dia_percentual is None:
                juros = Decimal("0")
            else:
                taxa = (cdi_dia_percentual / Decimal("100")) * percentual_cdi
                juros = saldo * taxa

                if arredondar_diariamente:
                    juros = dinheiro(juros)

                saldo = saldo + juros

                if arredondar_diariamente:
                    saldo = dinheiro(saldo)
        else:
            data_cdi = None
            cdi_dia_percentual = None
            juros = Decimal("0")

        linhas.append({
            "data": data,
            "data_cdi_usada": data_cdi,
            "saldo_abertura": float(saldo_abertura),
            "cdi_percentual_dia": float(cdi_dia_percentual) if cdi_dia_percentual is not None else None,
            "juros": float(juros),
            "saldo_fechamento": float(saldo)
        })

        data += timedelta(days=1)

    return pd.DataFrame(linhas)


cenarios = []

for deslocamento in [0, 1, 2, 3]:
    for arredondar in [False, True]:
        df_teste = calcular_com_deslocamento_cdi(deslocamento_dias=deslocamento, arredondar_diariamente=arredondar)

        saldo_final = dinheiro(Decimal(str(df_teste.iloc[-1]["saldo_fechamento"])))

        cenarios.append({
            "deslocamento_cdi": f"D-{deslocamento}",
            "arredondar_diariamente": arredondar,
            "saldo_final": saldo_final,
            "diferenca_extrato": saldo_final - valor_esperado_extrato,
        })

pd.DataFrame(cenarios)

,deslocamento_cdi,arredondar_diariamente,saldo_final,diferenca_extrato
0,D-0,False,868462.39,114.49
1,D-0,True,868462.47,114.57
2,D-1,False,868348.18,0.28
3,D-1,True,868348.14,0.24
4,D-2,False,868233.97,-113.93
5,D-2,True,868233.84,-114.06
6,D-3,False,868233.97,-113.93
7,D-3,True,868233.84,-114.06
